# LLM-RecFusion — Task 3: Faiss 向量离线建库与在线检索

**Objective**: 模拟将双塔模型（Dual-Tower）提取出的物品稠密向量灌入 Faiss 数据库，构建 ANN（近似最近邻）索引，并测试毫秒级的在线检索性能。

### 为什么需要 Faiss？

> 在工业级推荐系统中，物品库规模通常在百万到亿级别。双塔模型虽然能产出高质量的语义向量，但**暴力遍历**所有物品计算相似度在在线推理时不可接受。Faiss（Facebook AI Similarity Search）通过**倒排索引（IVF）**、**乘积量化（PQ）** 等算法，将检索延迟压榨到毫秒级，是业界标配的 ANN 引擎。

### 本 Notebook 流程图

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────────┐
│  模拟数据生成    │ --> │  Faiss 离线建库    │ --> │  在线检索 & Benchmark │
│ 100k 物品向量    │     │  IndexIVFFlat     │     │  Top-100 检索        │
│ 1 个用户向量     │     │  nlist=100        │     │  耗时统计            │
└─────────────────┘     └──────────────────┘     └─────────────────────┘
```

---


## 1. 导入依赖

> 确保环境中已安装 `faiss-cpu`、`numpy`、`pandas`。`time` 是 Python 内置模块，无需额外安装。

In [ ]:
# ============================================================
# Cell 1: 导入依赖
# ============================================================
import time
import warnings

import faiss
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print(f"Faiss version: {faiss.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. 模拟数据生成（Mock Data）

### 背景说明
- 模拟双塔模型**物品塔**离线产出的全量物品向量，共 100,000 条，维度 `dim=128`。
- 模拟用户实时请求时，**用户塔**前向传播在线产出的 1 条用户向量，维度 `dim=128`。
- 所有向量均做 **L2 归一化**，使得后续 Faiss 的**内积（Inner Product）检索等价于余弦相似度检索**。

> **为什么 L2 归一化后内积 = 余弦相似度？** 余弦相似度公式为 `cos(a,b) = a·b / (||a|| * ||b||)`，当向量被归一化到单位长度（L2 norm = 1）时，分母为 1，余弦相似度 = 点积。

In [ ]:
# ============================================================
# Cell 2: 模拟数据生成
# ============================================================

# 设定随机种子，保证结果可复现
np.random.seed(42)

# 参数配置
DIM = 128          # 向量维度
NUM_ITEMS = 100000 # 物品数量

# --- 2.1 生成物品向量 ---
# 从标准正态分布采样，生成 100000 x 128 的随机矩阵
item_embeddings = np.random.randn(NUM_ITEMS, DIM).astype(np.float32)

# L2 归一化：使每个物品向量的 L2 范数为 1
# axis=1 表示按行归一化（每个向量独立归一化）
item_embeddings = item_embeddings / np.linalg.norm(item_embeddings, axis=1, keepdims=True)

# --- 2.2 生成用户向量 ---
# 同样从标准正态分布采样
user_embedding = np.random.randn(1, DIM).astype(np.float32)

# L2 归一化
user_embedding = user_embedding / np.linalg.norm(user_embedding, axis=1, keepdims=True)

# --- 2.3 验证数据 ---
print(f"物品向量矩阵形状: {item_embeddings.shape}")
print(f"用户向量形状: {user_embedding.shape}")
print(f"数据类型: {item_embeddings.dtype}")
print(f"物品向量 L2 范数（检查前5条）: "
      f"{np.round(np.linalg.norm(item_embeddings[:5], axis=1), 6)}")
print(f"用户向量 L2 范数: {np.round(np.linalg.norm(user_embedding), 6)}")
print(f"\n✅ 数据生成完成！共 {NUM_ITEMS:,} 个物品向量，维度 {DIM}，均已 L2 归一化。")

## 3. 构建 Faiss 离线索引

### 索引选型策略（体现大厂工程思维）

| 索引类型 | 特点 | 适用场景 |
|---------|------|---------|
| `IndexFlatIP` | 暴力检索，精度 100% | 小规模（<1万）或精度验证 |
| `IndexIVFFlat` | 倒排索引 + 精确向量，**精度-速度可调** | **百万级召回，工业界标配** |
| `IndexIVFPQ` | 倒排索引 + 乘积量化，大幅压缩内存 | 十亿级，需要降低显存占用 |

> **本方案**：使用 `IndexFlatIP` 作为底层量化器（quantizer），在其上包裹 `IndexIVFFlat`。`nlist=100` 表示将 100,000 个物品向量聚成 100 个簇（centroids），检索时只需搜索 `nprobe` 个最近的簇，大幅减少计算量。

In [ ]:
# ============================================================
# Cell 3: 构建 Faiss 离线索引
# ============================================================

# --- 3.1 定义索引参数 ---
NLIST = 100   # 倒排索引聚类中心数量（簇数）
NPROBE = 10   # 检索时搜索的簇数量（精度-速度平衡参数）

# --- 3.2 构建基础索引（精确暴力检索） ---
# IndexFlatIP：内积索引，由于向量已 L2 归一化，内积 = 余弦相似度
quantizer = faiss.IndexFlatIP(DIM)

# --- 3.3 用倒排索引包裹基础索引 ---
# IndexIVFFlat(quantizer, d, nlist, metric)
# - quantizer: 基础索引，用于对簇心进行精确检索
# - d: 向量维度
# - nlist: 聚类中心数
# - metric: 距离度量，默认 METRIC_INNER_PRODUCT
index = faiss.IndexIVFFlat(quantizer, DIM, NLIST, faiss.METRIC_INNER_PRODUCT)

# --- 3.4 训练索引 ---
# IVF 需要训练（K-Means 聚类），学习数据分布以构建倒排表
print("正在训练 Faiss 索引（K-Means 聚类）...")
t_start = time.time()
index.train(item_embeddings)
t_train = time.time() - t_start
print(f"索引训练完成，耗时: {t_train:.4f} 秒")

# --- 3.5 添加数据 ---
# 将物品向量加入索引，建立倒排表
print("正在向索引添加物品向量...")
t_start = time.time()
index.add(item_embeddings)
t_add = time.time() - t_start
print(f"数据添加完成，耗时: {t_add:.4f} 秒")

# --- 3.6 设置检索参数 ---
# nprobe：检索时搜索的簇数量，值越大精度越高但速度越慢
# 这是控制精度与速度平衡的核心参数
index.nprobe = NPROBE

# --- 3.7 打印索引统计信息 ---
print(f"\n{'='*50}")
print(f"📊 Faiss 索引构建完成！")
print(f"{'='*50}")
print(f"索引类型: {type(index).__name__}")
print(f"向量维度: {DIM}")
print(f"物品总数: {index.ntotal:,}")
print(f"聚类中心数 (nlist): {NLIST}")
print(f"检索簇数 (nprobe): {NPROBE}")
print(f"是否已训练: {index.is_trained}")
print(f"{'='*50}")

## 4. 在线检索测试与耗时统计（Benchmark）

### 检索流程
1. 用户塔在线产出用户向量（已 L2 归一化）
2. 调用 `index.search()` 在 Faiss 索引中检索 Top-100 个最相似物品
3. 使用 `time.perf_counter()` 精确统计检索耗时
4. 用 Pandas 展示 Top-5 结果

> **关键词说明**：
> - `nprobe`：检索时搜索的倒排簇数量。例如 nlist=100 时，nprobe=10 表示只搜索 100 个簇中最相近的 10 个簇，计算量减少约 90%。
> - 工业实践中，通常设置 `nprobe ≈ sqrt(nlist)` 作为起点，再根据 Recall 指标微调。

In [ ]:
# ============================================================
# Cell 4: 在线检索测试与耗时统计
# ============================================================

K = 100  # 检索 Top-K 个最相似的物品

# --- 4.1 预热 ---
# 第一次检索可能包含 JIT 编译或缓存加载的开销，预热后统计更准确
_, _ = index.search(user_embedding, K)

# --- 4.2 正式检索：精确统计耗时 ---
# 使用 time.perf_counter() 获取高精度计时
t_start = time.perf_counter()
similarity_scores, item_indices = index.search(user_embedding, K)
t_elapsed = time.perf_counter() - t_start

# --- 4.3 转换结果 ---
# faiss.search 返回的形状：
# - similarity_scores: (1, K)，内积得分（因 L2 归一化，等价于余弦相似度）
# - item_indices: (1, K)，物品在原始数组中的索引
scores = similarity_scores[0]   # 形状转为 (K,)
indices = item_indices[0]       # 形状转为 (K,)

# --- 4.4 打印检索时间统计 ---
print(f"{'='*60}")
print(f"🚀 在线检索 Benchmark")
print(f"{'='*60}")
print(f"检索设置: Top-{K} | nprobe={NPROBE} | 物品总量={NUM_ITEMS:,}")
print(f"检索耗时: {t_elapsed * 1000:.4f} ms")  # 转换为毫秒
print()
print(">>> 通过 Faiss 的倒排索引，我们将几十万物品的语义检索延迟压榨到了毫秒级")
print(f"    （当前 {t_elapsed * 1000:.2f} ms），完美突破了大模型在推荐系统中的在线推理瓶颈。")
print()
print(f"每秒可处理查询数 (QPS): {1 / t_elapsed:,.0f}")
print(f"{'='*60}")

## 5. 检索结果展示（Top-5）

使用 Pandas 展示**最相似的 Top-5 物品**及其余弦相似度得分。

> 由于向量已 L2 归一化，Faiss 返回的内积得分直接就是余弦相似度，取值范围为 [-1, 1]。值越接近 1，表示物品与用户兴趣越相似。
> 在真实场景中，这些 `item_id` 可被送入排序层（Ranking Stage）做进一步精排。

In [ ]:
# ============================================================
# Cell 5: 使用 Pandas 展示 Top-5 结果
# ============================================================

# 构建结果 DataFrame
top5_df = pd.DataFrame({
    "Rank": range(1, 6),                           # 排名 1~5
    "Item_ID": indices[:5],                         # 物品在数组中的索引
    "Cosine_Similarity": np.round(scores[:5], 6)   # 余弦相似度（保留6位小数）
})

# 打印结果表格
print("📋 检索结果 Top-5 展示")
print("=" * 50)
print(top5_df.to_string(index=False))
print("=" * 50)
print()

# --- 5.1 简单统计结论 ---
print("📊 结论分析：")
print(f"  - Top-1 相似度: {scores[0]:.6f}")
print(f"  - Top-5 平均相似度: {scores[:5].mean():.6f}")
print(f"  - Top-100 平均相似度: {scores.mean():.6f}")
print(f"  - Top-100 相似度区间: [{scores.min():.6f}, {scores.max():.6f}]")

## 6. 进阶探讨（工业落地视角）

### 6.1 精度-速度权衡分析

`nprobe` 是 IVF 索引最重要的调优参数：
- `nprobe=1`：最快但召回率最低，仅搜索 1 个簇
- `nprobe=nlist`：搜索全部簇，退化为基础 Flat 检索的精度和速度
- **推荐起始值**：`nprobe ≈ sqrt(nlist)`，即 `nprobe=10` 对于 `nlist=100`

### 6.2 扩展到百万/亿级规模

| 规模 | 推荐索引 | 显存估算 | 延迟预期 |
|------|---------|---------|---------|
| 10万级 | IndexIVFFlat (nlist=100~500) | ~50 MB | <1 ms |
| 百万级 | IndexIVFFlat (nlist=1000~5000) | ~500 MB | 1-5 ms |
| 千万级 | IndexIVFPQ (M=64, nlist=10000) | ~200 MB | 5-20 ms |
| 亿级 | IndexIVFPQ + 多GPU | ~2 GB | 10-50 ms |

### 6.3 生产化建议
1. **内存索引**：将 Faiss 索引加载到共享内存中，所有推理容器共享同一份索引
2. **周期性重建**：物品向量更新后，每隔 T 小时（如 6h/24h）重建一次索引
3. **增量更新**：少量新增物品可用 `index.add()` 增量添加，但大量更新建议全量重建以防止簇分布偏移
4. **多索引路由**：不同内容类目（视频/图文/商品）可构建独立 Faiss 索引，按请求类型路由

---

✅ **Task 3 完成！Faiss 向量离线建库与在线检索链路已成功跑通。**

> 下一阶段：将 Faiss 的召回结果与 ALS 基线、双塔模型结果一起送入**多路召回融合模块**。